In [2]:
"""
06_imbalance_experiments.ipynb

Purpose:
Check whether imbalance-handling methods improve validation performance
for telecom fault severity classification.

Main idea:
Compare no resampling vs resampling methods using only the training split,
then evaluate on the untouched validation split.

Compared samplers:
1. none
2. RandomOverSampler
3. SMOTE

Models:
- LightGBM (primary)
- Random Forest (secondary)

Why:
LightGBM is the strongest model.
Random Forest also performed well, so it is worth testing.

"""

# 1. Imports

# sys is used only to print Python version
import sys

# Path helps create clean file paths
from pathlib import Path

# warnings are hidden to keep notebook output cleaner
import warnings

# joblib is used to save trained experiment models
import joblib

# numpy and pandas are used for data handling
import numpy as np
import pandas as pd

# Metrics used to compare experiments
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)

# Used for multiclass ROC-AUC
from sklearn.preprocessing import label_binarize

# Random Forest model
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# display() looks nicer in notebooks, but print() is used if unavailable
try:
    from IPython.display import display
except Exception:
    display = None

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)


# 2. Optional Imports

# LightGBM and imbalanced-learn may not always be installed
HAS_LIGHTGBM = True
HAS_IMBLEARN = True

try:
    from lightgbm import LGBMClassifier
except Exception:
    HAS_LIGHTGBM = False
    print("LightGBM not available.")

try:
    from imblearn.over_sampling import RandomOverSampler, SMOTE
except Exception:
    HAS_IMBLEARN = False
    print("imbalanced-learn not available. Install imbalanced-learn first.")


# 3. Paths

PROJECT_ROOT = Path.cwd().parent
SPLITS_PATH = PROJECT_ROOT / "data" / "splits"

REPORTS_PATH = PROJECT_ROOT / "reports" / "imbalance_experiments"
MODELS_PATH = PROJECT_ROOT / "artifacts" / "imbalance_experiments"

REPORTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SPLITS_PATH  :", SPLITS_PATH)
print("REPORTS_PATH :", REPORTS_PATH)
print("MODELS_PATH  :", MODELS_PATH)


# 4. Helper Functions

def show_df(df: pd.DataFrame, n: int = 5) -> None:
    """
    Show first few rows of a dataframe.
    """
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n))


def save_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    """
    Save dataframe as CSV.
    """
    df.to_csv(path, index=index)


def class_distribution_df(y, split_name):
    """
    Build a small class distribution table.

    This is useful to show:
    - original training imbalance
    - class balance after resampling
    """
    y = pd.Series(y)
    counts = y.value_counts().sort_index()
    props = y.value_counts(normalize=True).sort_index().round(4)

    return pd.DataFrame(
        {
            "split": split_name,
            "class": counts.index,
            "count": counts.values,
            "proportion": props.values,
        }
    )


def evaluate_multiclass(y_true, y_pred, y_proba, class_labels):
    """
    Compute main multi-class evaluation metrics.

    Macro F1 is the main metric because the data is imbalanced
    and we want fair performance across classes.
    """
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    # Convert multiclass labels to binary form for ROC-AUC
    y_true_bin = label_binarize(y_true, classes=class_labels)

    result["roc_auc_ovr_macro"] = roc_auc_score(
        y_true_bin,
        y_proba,
        multi_class="ovr",
        average="macro",
    )

    result["roc_auc_ovr_weighted"] = roc_auc_score(
        y_true_bin,
        y_proba,
        multi_class="ovr",
        average="weighted",
    )

    return result


def get_default_models():
    """
    Define the models to test in imbalance experiments.

    These are based on your stronger tuned settings.
    """
    models = {}

    # Random Forest with tuned-like settings
    models["random_forest"] = RandomForestClassifier(
        n_estimators=200,
        min_samples_split=10,
        min_samples_leaf=1,
        max_features=None,
        max_depth=40,
        bootstrap=True,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    # LightGBM with tuned-like settings
    if HAS_LIGHTGBM:
        models["lightgbm"] = LGBMClassifier(
            subsample=0.9,
            reg_lambda=0.01,
            reg_alpha=0.1,
            num_leaves=31,
            n_estimators=700,
            min_child_samples=30,
            max_depth=20,
            learning_rate=0.03,
            colsample_bytree=0.7,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            verbosity=-1,
        )

    return models


def get_sampler_dict():
    """
    Define the resampling methods to compare.

    none = no resampling
    random_oversampler = randomly duplicate minority samples
    smote = create synthetic minority samples
    """
    samplers = {"none": None}

    if HAS_IMBLEARN:
        samplers["random_oversampler"] = RandomOverSampler(random_state=RANDOM_STATE)
        samplers["smote"] = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)

    return samplers


# 5. Load Splits

# Load train and validation feature sets
X_train = pd.read_csv(SPLITS_PATH / "X_train.csv")
X_valid = pd.read_csv(SPLITS_PATH / "X_valid.csv")

# Load target labels
y_train = pd.read_csv(SPLITS_PATH / "y_train.csv")["fault_severity"]
y_valid = pd.read_csv(SPLITS_PATH / "y_valid.csv")["fault_severity"]

print("\nLoaded splits:")
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

# Get sorted class labels
class_labels = sorted(y_train.unique().tolist())
print("Classes:", class_labels)

# Show original training class imbalance
print("\nOriginal training class distribution:")
orig_train_dist = class_distribution_df(y_train, "train_original")
print(orig_train_dist)


# 6. Define Models and Samplers

models = get_default_models()
samplers = get_sampler_dict()

if len(models) == 0:
    raise ValueError("No usable models found.")

if len(samplers) == 0:
    raise ValueError("No samplers found.")

print("\nModels to evaluate:", list(models.keys()))
print("Samplers to evaluate:", list(samplers.keys()))


# 7. Run Imbalance Experiments

experiment_rows = []
per_class_rows = []
resampled_dist_rows = []

# Test every model with every sampler
for model_name, model in models.items():
    for sampler_name, sampler in samplers.items():
        experiment_name = f"{model_name}__{sampler_name}"
        print(f"\nRUNNING EXPERIMENT: {experiment_name}")

        # Apply resampling only to training data
        if sampler is None:
            X_train_res = X_train.copy()
            y_train_res = y_train.copy()
        else:
            X_train_res, y_train_res = sampler.fit_resample(X_train, y_train)

        print("Resampled X_train shape:", X_train_res.shape)
        print("Resampled y_train shape:", y_train_res.shape)

        # Save resampled class distribution
        dist_df = class_distribution_df(y_train_res, f"{experiment_name}_resampled")
        resampled_dist_rows.append(dist_df)

        # Train model on resampled training data
        model.fit(X_train_res, y_train_res)

        # Evaluate on untouched validation set
        y_pred = model.predict(X_valid)
        y_proba = model.predict_proba(X_valid)

        metrics = evaluate_multiclass(
            y_true=y_valid,
            y_pred=y_pred,
            y_proba=y_proba,
            class_labels=class_labels,
        )

        # Save experiment-level metrics
        row = {
            "model": model_name,
            "sampler": sampler_name,
            "experiment": experiment_name,
            "train_rows_after_resampling": len(X_train_res),
        }
        row.update({k: round(v, 6) for k, v in metrics.items()})
        experiment_rows.append(row)

        # Save per-class metrics
        clf_rep = classification_report(
            y_valid,
            y_pred,
            labels=class_labels,
            output_dict=True,
            zero_division=0,
        )

        for cls in class_labels:
            cls_key = str(cls)
            per_class_rows.append(
                {
                    "model": model_name,
                    "sampler": sampler_name,
                    "experiment": experiment_name,
                    "class": cls,
                    "precision": clf_rep[cls_key]["precision"],
                    "recall": clf_rep[cls_key]["recall"],
                    "f1_score": clf_rep[cls_key]["f1-score"],
                    "support": clf_rep[cls_key]["support"],
                }
            )

        # Save trained experiment model
        joblib.dump(model, MODELS_PATH / f"{experiment_name}.joblib")

        print(f"{experiment_name} complete.")


# 8. Summaries

# Overall experiment comparison
experiment_df = pd.DataFrame(experiment_rows).sort_values(
    by=["f1_macro", "balanced_accuracy", "roc_auc_ovr_macro"],
    ascending=False,
).reset_index(drop=True)

# Per-class metrics table
per_class_df = pd.DataFrame(per_class_rows)

# Resampled class distribution table
resampled_dist_df = pd.concat(resampled_dist_rows, axis=0, ignore_index=True)

print("\nIMBALANCE EXPERIMENT SUMMARY")
print(experiment_df)

save_csv(experiment_df, REPORTS_PATH / "imbalance_experiment_summary.csv", index=False)
save_csv(per_class_df, REPORTS_PATH / "imbalance_per_class_metrics.csv", index=False)
save_csv(resampled_dist_df, REPORTS_PATH / "resampled_class_distributions.csv", index=False)

if display is not None:
    display(experiment_df)
    display(per_class_df.head(18))
    display(resampled_dist_df.head(18))


# 9. Best Experiment

# Best experiment is the first row after sorting
best_row = experiment_df.iloc[0]

best_summary = pd.DataFrame(
    {
        "selection_rule": ["Highest validation Macro F1 after imbalance experiment"],
        "best_model": [best_row["model"]],
        "best_sampler": [best_row["sampler"]],
        "best_experiment": [best_row["experiment"]],
        "f1_macro": [best_row["f1_macro"]],
        "balanced_accuracy": [best_row["balanced_accuracy"]],
        "roc_auc_ovr_macro": [best_row["roc_auc_ovr_macro"]],
    }
)

print("\nBEST IMBALANCE EXPERIMENT")
print(best_summary)

save_csv(best_summary, REPORTS_PATH / "best_imbalance_experiment_summary.csv", index=False)


# 10. Compact Notebook Summary

# Small summary table 
summary_df = pd.DataFrame(
    {
        "item": [
            "n_train_rows_original",
            "n_valid_rows",
            "n_features",
            "n_models_tested",
            "n_samplers_tested",
            "primary_metric",
            "best_model",
            "best_sampler",
        ],
        "value": [
            len(X_train),
            len(X_valid),
            X_train.shape[1],
            len(models),
            len(samplers),
            "f1_macro",
            best_row["model"],
            best_row["sampler"],
        ],
    }
)

print("\nNOTEBOOK SUMMARY")
print(summary_df)

save_csv(summary_df, REPORTS_PATH / "imbalance_notebook_summary.csv", index=False)

print("\nIMBALANCE EXPERIMENTS COMPLETE")
print("Reports saved to :", REPORTS_PATH)
print("Models saved to  :", MODELS_PATH)

Python : 3.12.2
Pandas : 2.3.3
NumPy  : 2.3.5
PROJECT_ROOT : /Users/hasheenadilmidesilva/Desktop/NetGuard
SPLITS_PATH  : /Users/hasheenadilmidesilva/Desktop/NetGuard/data/splits
REPORTS_PATH : /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/imbalance_experiments
MODELS_PATH  : /Users/hasheenadilmidesilva/Desktop/NetGuard/artifacts/imbalance_experiments

Loaded splits:
X_train: (4723, 860)
X_valid: (1181, 860)
y_train: (4723,)
y_valid: (1181,)
Classes: [0, 1, 2]

Original training class distribution:
            split  class  count  proportion
0  train_original      0   3061      0.6481
1  train_original      1   1197      0.2534
2  train_original      2    465      0.0985

Models to evaluate: ['random_forest', 'lightgbm']
Samplers to evaluate: ['none', 'random_oversampler', 'smote']

RUNNING EXPERIMENT: random_forest__none
Resampled X_train shape: (4723, 860)
Resampled y_train shape: (4723,)
random_forest__none complete.

RUNNING EXPERIMENT: random_forest__random_oversampler
Resam

,model,sampler,experiment,train_rows_after_resampling,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr_macro,roc_auc_ovr_weighted
0,lightgbm,none,lightgbm__none,4723,0.744285,0.739017,0.690128,0.739017,0.707817,0.773797,0.744285,0.752550,0.882571,0.860530
1,random_forest,none,random_forest__none,4723,0.744285,0.734378,0.677128,0.734378,0.700061,0.764286,0.744285,0.750496,0.888579,0.867674
2,lightgbm,random_oversampler,lightgbm__random_oversampler,9183,0.745131,0.720064,0.681257,0.720064,0.696189,0.768511,0.745131,0.752529,0.885648,0.865777
3,lightgbm,smote,lightgbm__smote,9183,0.756986,0.700490,0.688522,0.700490,0.694254,0.758867,0.756986,0.757831,0.884816,0.864993
4,random_forest,smote,random_forest__smote,9183,0.732430,0.703300,0.657342,0.703300,0.676232,0.741595,0.732430,0.735483,0.881619,0.859866
5,random_forest,random_oversampler,random_forest__random_oversampler,9183,0.722269,0.702597,0.650786,0.702597,0.670622,0.748697,0.722269,0.730658,0.880155,0.859503


,model,sampler,experiment,class,precision,recall,f1_score,support
0,random_forest,none,random_forest__none,0,0.863636,0.768930,0.813536,766.0
1,random_forest,none,random_forest__none,1,0.577143,0.675585,0.622496,299.0
2,random_forest,none,random_forest__none,2,0.590604,0.758621,0.664151,116.0
3,random_forest,random_oversampler,random_forest__random_oversampler,0,0.860778,0.750653,0.801953,766.0
4,random_forest,random_oversampler,random_forest__random_oversampler,1,0.536785,0.658863,0.591592,299.0
5,random_forest,random_oversampler,random_forest__random_oversampler,2,0.554795,0.698276,0.618321,116.0
6,random_forest,smote,random_forest__smote,0,0.831956,0.788512,0.809651,766.0
7,random_forest,smote,random_forest__smote,1,0.580858,0.588629,0.584718,299.0
8,random_forest,smote,random_forest__smote,2,0.559211,0.732759,0.634328,116.0
9,lightgbm,none,lightgbm__none,0,0.880916,0.753264,0.812104,766.0


,split,class,count,proportion
0,random_forest__none_resampled,0,3061,0.6481
1,random_forest__none_resampled,1,1197,0.2534
2,random_forest__none_resampled,2,465,0.0985
3,random_forest__random_oversampler_resampled,0,3061,0.3333
4,random_forest__random_oversampler_resampled,1,3061,0.3333
5,random_forest__random_oversampler_resampled,2,3061,0.3333
6,random_forest__smote_resampled,0,3061,0.3333
7,random_forest__smote_resampled,1,3061,0.3333
8,random_forest__smote_resampled,2,3061,0.3333
9,lightgbm__none_resampled,0,3061,0.6481



BEST IMBALANCE EXPERIMENT
                                      selection_rule best_model best_sampler best_experiment  f1_macro  balanced_accuracy  roc_auc_ovr_macro
0  Highest validation Macro F1 after imbalance ex...   lightgbm         none  lightgbm__none  0.707817           0.739017           0.882571

NOTEBOOK SUMMARY
                    item     value
0  n_train_rows_original      4723
1           n_valid_rows      1181
2             n_features       860
3        n_models_tested         2
4      n_samplers_tested         3
5         primary_metric  f1_macro
6             best_model  lightgbm
7           best_sampler      none

IMBALANCE EXPERIMENTS COMPLETE
Reports saved to : /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/imbalance_experiments
Models saved to  : /Users/hasheenadilmidesilva/Desktop/NetGuard/artifacts/imbalance_experiments
